# 13주차 — 실전 — 코호트 추출과 Table 1
**의료진을 위한 데이터베이스** · 미니 EMR 실습 노트북

사용법: 위에서부터 순서대로 셀을 실행하세요 (`Shift+Enter`).
`q("SELECT ...")` 안의 SQL만 바꿔가며 실험하면 됩니다.
어떤 셀이 오류로 막히거나 데이터가 엉켰다면, 아무 셀에서 `reset_db()`를 실행하면 처음 상태로 돌아갑니다.

In [ ]:
# [준비] 미니 EMR 데이터베이스 내려받기 + 도우미 정의 — 이 셀을 가장 먼저 실행!
import os, shutil, sqlite3, urllib.request
import pandas as pd

DB_URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/mini_emr/mini_emr.sqlite"
DB_PATH = "mini_emr.sqlite"
WORK_PATH = DB_PATH   # 실습에 사용할 파일

con = None

def reset_db():
    """DB를 원본 상태로 되돌리고 다시 연결 (막히면 언제든 호출)"""
    global con
    if con is not None:
        con.close()
    try:
        urllib.request.urlretrieve(DB_URL, DB_PATH)
    except Exception as e:
        if not os.path.exists(DB_PATH):
            raise RuntimeError(
                "DB 다운로드 실패 — 왼쪽 파일 탭에 mini_emr.sqlite를 업로드한 뒤 이 셀을 다시 실행하세요."
            ) from e
        print("다운로드 실패 — 업로드되어 있는 파일을 사용합니다.")
    if WORK_PATH != DB_PATH:
        shutil.copyfile(DB_PATH, WORK_PATH)   # 원본 보존, 실습은 사본에서
    con = sqlite3.connect(WORK_PATH)
    con.execute("PRAGMA foreign_keys = ON")   # SQLite는 기본으로 꺼져 있음 — 반드시 켠다
    assert con.execute("PRAGMA foreign_keys").fetchone()[0] == 1

def q(sql):
    """조회(SELECT) 실행 → 결과를 표로 반환"""
    return pd.read_sql_query(sql, con)

def run(sql):
    """데이터/구조를 바꾸는 SQL(INSERT/UPDATE/DELETE/CREATE 등) 실행"""
    con.executescript(sql); con.commit(); print("실행 완료")

reset_db()
q("SELECT name AS 테이블 FROM sqlite_master WHERE type='table' ORDER BY 1")

## 연구 질문
**"2020년 1월 1일 이후 처음으로 뇌경색(I63)을 진단받은 환자"** 코호트를 만들고 Table 1까지 완성한다.

## 1단계 — 후보군: I63 이력자 전체와 첫 진단일

In [ ]:
# 첫 진단일 (washout 적용 전 — 10명)
q("""
SELECT v.patient_id, MIN(v.visit_date) AS first_dx_date
FROM visit v JOIN diagnosis d ON v.visit_id = d.visit_id
WHERE d.kcd_code LIKE 'I63%'
GROUP BY v.patient_id ORDER BY first_dx_date
""")

## 2단계 — 신규 환자 제한: 관찰된 '첫' 진단이 2020년 이후
엄밀한 washout은 index date 이전 일정 기간(look-back)의 이력 부재를 확인하는 것입니다. 이 데이터셋은 관찰이 2019년부터 시작되므로, `MIN(진단일) >= '2020-01-01'`은 **약 1년의 look-back**에 해당하는 간이 조건입니다 — 정의서에는 이런 한계를 명시해야 합니다.

In [ ]:
# index date 확정 (8명 — P029·P030은 2019년 진단력으로 제외)
q("""
SELECT v.patient_id, MIN(v.visit_date) AS index_date
FROM visit v JOIN diagnosis d ON v.visit_id = d.visit_id
WHERE d.kcd_code LIKE 'I63%'
GROUP BY v.patient_id
HAVING MIN(v.visit_date) >= '2020-01-01'
ORDER BY index_date
""")

## 3단계 — WITH(CTE)로 특성 붙이기
나이는 연도 차이(연 나이) 근사입니다 — 생일 미도래 환자는 1세 크게 계산될 수 있습니다.

In [ ]:
# 코호트 + 성별 + index 시점 나이(연 나이 근사)
q("""
WITH stroke_cohort AS (
  SELECT v.patient_id, MIN(v.visit_date) AS index_date
  FROM visit v JOIN diagnosis d ON v.visit_id = d.visit_id
  WHERE d.kcd_code LIKE 'I63%'
  GROUP BY v.patient_id
  HAVING MIN(v.visit_date) >= '2020-01-01')
SELECT c.patient_id, c.index_date, p.sex,
       CAST(substr(c.index_date,1,4) AS INT)
         - CAST(substr(p.birth_date,1,4) AS INT) AS age_at_index
FROM stroke_cohort c JOIN patient p ON p.patient_id = c.patient_id
""")

## 4단계 — 기저 동반질환 플래그 (EXISTS)
기저(baseline) 동반질환이므로 **index date 이전(당일 포함)** 이력만 세야 합니다 — 시간 조건이 없으면 뇌경색 이후에 생긴 고혈압까지 '기저'로 잘못 집계됩니다. 아래는 이 강의의 조작적 정의(operational definition)이며, 실제 연구에서는 look-back 기간까지 명시합니다.

In [ ]:
# 고혈압(I10) 기저 동반 여부 — index date 이전 이력만
q("""
WITH stroke_cohort AS (
  SELECT v.patient_id, MIN(v.visit_date) AS index_date
  FROM visit v JOIN diagnosis d ON v.visit_id = d.visit_id
  WHERE d.kcd_code LIKE 'I63%'
  GROUP BY v.patient_id
  HAVING MIN(v.visit_date) >= '2020-01-01')
SELECT c.patient_id, c.index_date,
  CASE WHEN EXISTS (
    SELECT 1 FROM visit v2 JOIN diagnosis d2 ON d2.visit_id = v2.visit_id
    WHERE v2.patient_id = c.patient_id
      AND d2.kcd_code LIKE 'I10%'
      AND v2.visit_date <= c.index_date)
  THEN 1 ELSE 0 END AS htn_baseline
FROM stroke_cohort c
""")

## 5단계 — Table 1

In [ ]:
# 성별·평균나이 요약
q("""
WITH stroke_cohort AS (
  SELECT v.patient_id, MIN(v.visit_date) AS index_date
  FROM visit v JOIN diagnosis d ON d.visit_id = v.visit_id
  WHERE d.kcd_code LIKE 'I63%'
  GROUP BY v.patient_id
  HAVING MIN(v.visit_date) >= '2020-01-01')
SELECT p.sex AS 성별, COUNT(*) AS 명수,
  ROUND(AVG(CAST(substr(c.index_date,1,4) AS INT)
    - CAST(substr(p.birth_date,1,4) AS INT)), 1) AS 평균나이
FROM stroke_cohort c JOIN patient p ON p.patient_id = c.patient_id
GROUP BY p.sex
""")

## 프로젝트 스타터 — 이 뼈대를 복사해 조별 주제로 바꾸세요

In [ ]:
# [프로젝트 템플릿]
COHORT_SQL = """
WITH my_cohort AS (
  -- ① 포함 기준: 코드/기간을 조별 주제로 교체
  SELECT v.patient_id, MIN(v.visit_date) AS index_date
  FROM visit v JOIN diagnosis d ON d.visit_id = v.visit_id
  WHERE d.kcd_code LIKE '__%'          -- ← 여기
  GROUP BY v.patient_id
  HAVING MIN(v.visit_date) >= '2020-01-01'  -- ← washout 기준
)
SELECT c.*, p.sex FROM my_cohort c
JOIN patient p ON p.patient_id = c.patient_id
"""
# q(COHORT_SQL)  # 주석을 풀어 실행

---
## 정답 및 해설
먼저 스스로 푼 다음에 열어 보세요.

**프로젝트 체크리스트**
```sql
-- ① 포함 기준(코드·기간)과 제외 기준을 정의서에 먼저 글로 쓴다
-- ② 단계마다 대상자 수를 주석으로 남긴다 (attrition)
-- ③ Table 1 변수는 성별·연령 + 조별 선택 2개 이상
```